In [1]:
from Bio import AlignIO,SeqIO
import numpy as np
import pandas as pd
import re

# ASR tools functions

ASR tools are a series of modules used to analyse and process ancestral sequences and associated statistical supports from PAML rst files. Many of the tools are necessary for the processing of insertions and deletions. 

In [11]:
from modules.branch_mutations import branch_mutations
from modules.pd_post_dist import pd_post_dist
from modules.read_rst import read_rst
from modules.pd_ML_residues import pd_ML_residues
from modules.get_anc_trees import get_anc_trees

# Extraction of Anc trees from CodeML
The discrete trait reconstruction requires the Anc labelled tree from CodeML rst file with the correct branch lengths. This block of code will extract the correct Anc tree. 

In [4]:
reconstructions=[]
x=[]
trees = ['List of nwk tree files to process']


for i in trees:
    x=i.split('/')[0]
    reconstructions.append(x+'/rst')

for reconstruction,tree in zip(reconstructions,trees):
    
    anc_trees = get_anc_trees(reconstruction)
    print(anc_trees[0], file=open(str(tree)+'_anc.nwk', 'a'))
    print(anc_trees[1], file=open(str(tree)+'_anc_bl.nwk', 'a'))

# Gap/tip processing

This script will process the alignment, detect sites where at least 2% of sequences have a gap character and produce a dataframe of tips and whether the gap character is present at each site index. The output csv file should then be loaded into the R script for discrete character reconstruction. 

In [5]:
aln = AlignIO.read('Seqeuence alignment in fasta format', 'fasta')

tips = []
for i in aln:
    tips.append(str(i.id))

pd_list = []
ins_site = []
for i in aln:
    pd_list.append(list(i.seq))
pd_aln = pd.DataFrame(pd_list)
sites = pd_aln.columns
for i in sites:
    if i < 1:
        continue
    else:
        x = pd_aln[i].to_list()
        prop = x.count('-')/len(pd_aln)
        if prop > 0.01:
            ins_site.append(i)
            
ins_cols = []
for ins in ins_site:
    n = []
    n.append('insertion_'+str(ins))
    for i in aln:
        x = str(i.seq[ins])
        if x == '-':
            n.append('n')
        else:
            n.append('y')
    ins_cols.append(n)

    
ids = []
for i in aln:
    ids.append(i.id)
    
ins_pd = pd.DataFrame(ins_cols)
ins_pd = ins_pd.transpose()
new_cols = ins_pd.iloc[0]
ins_pd = ins_pd[1:]
ins_pd.columns = new_cols
ins_pd['id'] = ids
cols = ins_pd.columns.tolist()
cols = cols[-1:] + cols[:-1]
ins_pd = ins_pd[cols]
ins_pd.to_csv('Insertion_tip_dataframe.csv')


,id,insertion_1,insertion_2,insertion_3,insertion_4,insertion_5,insertion_6,insertion_7,insertion_8,insertion_9,...,insertion_823,insertion_824,insertion_825,insertion_826,insertion_827,insertion_828,insertion_833,insertion_834,insertion_835,insertion_836
1,A0A011MFK6,n,n,n,n,n,n,n,n,n,...,y,y,y,y,y,y,n,n,n,n
2,A0A031I3F0,n,n,n,n,n,n,n,n,n,...,y,y,y,y,y,y,n,n,n,n
3,A0A059KPS2,n,y,n,n,n,n,n,n,n,...,y,y,y,y,y,y,n,n,n,n
4,A0A059KR93,n,y,n,n,n,n,n,n,n,...,y,y,y,y,y,y,n,n,n,n
5,A0A073JDY3,n,n,n,n,n,n,n,n,n,...,y,y,y,y,y,y,n,n,n,n
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
431,A0A6M5XXN9,n,y,n,n,n,n,n,n,n,...,y,y,y,y,y,y,n,n,n,n
432,A0A6P1SBG6,n,n,n,n,n,n,n,n,n,...,y,y,y,y,y,y,n,n,n,n
433,A0A7C4E7E0,n,n,n,n,n,n,n,n,n,...,y,y,y,y,y,y,n,n,n,n
434,I4MYG7,n,n,n,n,n,n,n,n,n,...,y,y,n,n,n,n,n,n,n,n


## Stop running code here. Run the R code in the discrete_trait_reconstruction notebook before returning to the below cells. 

# If ancestral sequences are generated by GRASP or IQ-TREE (do not require extraction from CodeML RST file)
Once the discrete character reconstruction has been run in the associated iR notebook, unprocessed ancestral sequences and insertion probabilities can be loaded in as a Seq object and a pandas dataframe. The below code will remove residues where a gap character is reconstructed with a posterior probability greater than or equal to 0.5. 

In [3]:
anc_aln=AlignIO.read('unprocessed_ancs.fasta','fasta')
ins_prob=pd.read_csv('Insertion_anc_probabilities_dataframe.csv')

In [53]:
nodes_old=ins_prob['node'].tolist()
new_nodes=[]
for i in nodes_old:
    if str(i)=='nan':
        new_nodes.append(str(i))
    elif i=='Root':
        new_nodes.append(i)
    else:
        new_nodes.append(i.split('/')[0])
ins_prob['node']=new_nodes
for seq in anc_aln:
    for i,n in enumerate(new_nodes):
        if seq.id==n:
            prob_dist=ins_prob.iloc[i]
            prob_dist=prob_dist[2:]
            indexes=prob_dist.index.tolist()
            probabilities=prob_dist.tolist()
            sites=[]
            for f in indexes:
                sites.append(f.split('_')[1])
            corrected_seq=[]
            for z,(prob,idx,aa) in enumerate(zip(probabilities,indexes,seq.seq)):
                if prob >= 0.5:
                    corrected_seq.append('-')
                else:
                    corrected_seq.append(aa)
            print('>'+str(seq.id),file=open('Processed_ancs.fasta','a'))
            t=''.join(corrected_seq)
            print(t, file=open('Processed_ancs.fasta','a'))

# If ancestral sequences are reconstructed in CodeML and require extraction from a PAML rst file
The below code will extract sequences from the rst file using read_rst and pd_post_dist functions defined above. Note that this code was written for multiple replicates of reconstruction/tree search. Input trees can be a single replicate, however just need to be a 1 element list defined as the "trees" variable on line 12, and a single rst file defined as the variable "reconstructions" on line 11. Note that sites can be manually dropped using the sites_to_drop variable on line 9.  

In [9]:
ins_pd = pd.read_csv('Insertion_tip_dataframe.csv')
ins_list = ins_pd.columns.tolist()
ins_list = ins_list[2:]
site = []
for i in ins_list:
    x = i.split('_')
    site.append(int(x[1]))

sites_to_drop = [0]

for reconstruction,tree in zip(reconstructions,trees):

    infile=reconstruction
    nodes=list(range(436,868))
    seq_length=838

    for i, node in enumerate(nodes):
        rst = read_rst(infile, node_point=int(node), seq_length=seq_length)
        df = pd_post_dist(rst)
        df = df.astype(float)
        df[20] = df.idxmax(axis=1)
        df.columns = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K',
                      'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V','res_idx']
        idx_to_res = {}
        for n,q in enumerate(df.columns):
            if q == 'res_idx':
                continue
            else:
                idx_to_res[n] = q
        df['res'] = df['res_idx'].map(idx_to_res)
        ML_res = df['res'].to_list()
        df = df.drop(['res_idx', 'res'], axis=1)
        df['ML'] = df.max(axis =1)
        ML_prob = []
        ML_prob = df['ML'].to_list()
        if i == 0:
            ml_df = pd.DataFrame(ML_prob).transpose()
            res_df = pd.DataFrame(ML_res).transpose()
        else:
            ml_df = pd.concat([ml_df, pd.DataFrame(ML_prob).transpose()], axis = 0)
            res_df = pd.concat([res_df, pd.DataFrame(ML_res).transpose()], axis = 0)


        insertion_mapping = pd.read_csv(str(tree)+'_insertion_mapping.csv')
        insertion_mapping = insertion_mapping.drop(['Unnamed: 0'], axis = 1)
        node_list = insertion_mapping['node'].to_list()
        for w,x in enumerate(node_list):
            if x == 'Root':
                continue
            elif int(x) == int(node):
                ins_index = insertion_mapping.values.tolist()[w]


        for o,y in zip(ins_index[1:], site):
            if float(o) >= 0.5:
                ml_df.iloc[i,y] = np.nan
                res_df.iloc[i,y] = '-'
    
    ml_df.columns = list(range(1, seq_length+1))
    res_df.columns = list(range(1, seq_length+1))
    ml_df.index = [nodes]
    res_df.index = [nodes]  
    ml_df['mean'] = ml_df.mean(numeric_only=True, axis=1)
    ml_df.to_csv(str(reconstruction)+'_posterior_dist.csv')
    for l,k in zip(nodes, list(range(len(res_df)))):
        seq_list = res_df.values.tolist()[k]
        seq = ''.join(seq_list)
        print('>'+str(l), file = open(str(reconstruction)+'_anc_seqs.txt', 'a'))
        print(seq, file = open(str(reconstruction)+'_anc_seqs.txt', 'a'))
    print('Finished '+str(reconstruction))

Finished 1/rst
Finished 2/rst
Finished 3/rst
Finished 4/rst
Finished 5/rst
Finished 6/rst
Finished 7/rst
Finished 8/rst
Finished 11/rst
Finished 12/rst
Finished 15/rst
Finished 16/rst
Finished 17/rst
Finished 18/rst
Finished 20/rst
Finished 21/rst
Finished 22/rst
Finished 24/rst
Finished 25/rst
Finished 26/rst
Finished 27/rst
Finished 28/rst
Finished 29/rst
Finished 30/rst
Finished 31/rst
Finished 33/rst
Finished 34/rst
Finished 35/rst
Finished 36/rst
Finished 37/rst
Finished 38/rst
Finished 39/rst
Finished 40/rst
Finished 41/rst
Finished 42/rst
Finished 43/rst
Finished 44/rst
Finished 45/rst
Finished 46/rst
Finished 47/rst
Finished 48/rst
Finished 49/rst
Finished 50/rst
Finished 52/rst
